In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# User input
dataset_path = r"Dataset\comparison_outputs\Movies-Dataset-no-missing-2000-2026-ML-Ready.csv"
numerical_features = [
    "budget",
    "runtime",
    "actor_1_popularity",
    "actor_2_popularity",
    "actor_3_popularity",
    "revenue",
]
multiLabelCategorical_features = [
    "production_countries",
    "production_companies",
    "genres",
    "spoken_languages",
]
OneHotEncodingCategorical_features = [
    "original_language",
    "mpaa_rating",
]
TextBased_features = ["keywords", "tagline", "overview"]
passthrough_features = ["title", "release_date", *numerical_features]

DATA_PATH = Path(dataset_path)
OUTPUT_PATH = DATA_PATH.parent / f"{DATA_PATH.stem}-encoded.csv"

# Protect common studio names that contain hyphens so they are not split into fake companies.
PROTECTED_COMPANY_NAMES = [
    "Metro-Goldwyn-Mayer",
    "Craven-Maddalena Films",
    "Davis-Panzer Productions",
    "Tele M\u00fcnchen Fernseh Produktionsgesellschaft (TMG)",
]

MULTILABEL_FEATURE_CONFIGS = {
    "production_countries": {
        "column": "production_countries",
        "separator": ",",
        "prefix": "country_",
        "top_k": 10,
        "protected_values": None,
    },
    "production_companies": {
        "column": "production_companies",
        "separator": "-",
        "prefix": "studio_",
        "top_k": 20,
        "protected_values": PROTECTED_COMPANY_NAMES,
    },
    "genres": {
        "column": "genres",
        "separator": "-",
        "prefix": "genre_",
        "top_k": None,
        "protected_values": None,
    },
    "spoken_languages": {
        "column": "spoken_languages",
        "separator": ",",
        "prefix": "spoken_language_",
        "top_k": 5,
        "protected_values": None,
    },
}

ONEHOT_FEATURE_CONFIGS = {
    "original_language": {"column": "original_language"},
    "mpaa_rating": {"column": "mpaa_rating"},
}


class MultiLabelColumnEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column, separator, prefix, top_k=None, protected_values=None):
        self.column = column
        self.separator = separator
        self.prefix = prefix
        self.top_k = top_k
        self.protected_values = protected_values
        self._placeholder = "__PROTECTED_HYPHEN__"

    def _get_series(self, X):
        if isinstance(X, pd.DataFrame):
            series = X[self.column]
        elif isinstance(X, pd.Series):
            series = X
        else:
            series = pd.Series(np.asarray(X).ravel(), name=self.column)
        return series.fillna("").astype(str)

    def _split_labels(self, value):
        value = value.strip()
        if not value:
            return []

        for protected in self.protected_values or []:
            value = value.replace(protected, protected.replace("-", self._placeholder))

        labels = []
        for item in value.split(self.separator):
            cleaned = item.strip().replace(self._placeholder, "-")
            if cleaned:
                labels.append(cleaned)
        return labels

    def fit(self, X, y=None):
        series = self._get_series(X)
        labels = series.apply(self._split_labels)

        self.encoder_ = MultiLabelBinarizer()
        encoded = pd.DataFrame(
            self.encoder_.fit_transform(labels),
            columns=self.encoder_.classes_,
            index=series.index,
        )

        if self.top_k is None:
            self.selected_labels_ = list(encoded.columns)
        else:
            self.selected_labels_ = encoded.sum().nlargest(self.top_k).index.tolist()

        self.feature_names_ = [f"{self.prefix}{label}" for label in self.selected_labels_]
        return self

    def transform(self, X):
        series = self._get_series(X)
        labels = series.apply(self._split_labels)

        encoded = pd.DataFrame(
            self.encoder_.transform(labels),
            columns=self.encoder_.classes_,
            index=series.index,
        )
        encoded = encoded.reindex(columns=self.selected_labels_, fill_value=0)
        encoded.columns = self.feature_names_
        return encoded

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


class OneHotColumnEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column):
        self.column = column

    def _get_series(self, X):
        if isinstance(X, pd.DataFrame):
            series = X[self.column]
        elif isinstance(X, pd.Series):
            series = X
        else:
            series = pd.Series(np.asarray(X).ravel(), name=self.column)
        series = series.fillna("Unknown").astype(str).str.strip()
        return series.replace("", "Unknown")

    def fit(self, X, y=None):
        series = self._get_series(X)
        self.encoder_ = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=int)
        self.encoder_.fit(series.to_frame(name=self.column))
        self.feature_names_ = self.encoder_.get_feature_names_out([self.column]).tolist()
        return self

    def transform(self, X):
        series = self._get_series(X)
        encoded = self.encoder_.transform(series.to_frame(name=self.column))
        return pd.DataFrame(encoded, columns=self.feature_names_, index=series.index)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


class MovieTextEncoder(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        keyword_column="keywords",
        tagline_column="tagline",
        overview_column="overview",
        keyword_top_k=25,
        tfidf_max_features=500,
        n_components=5,
        random_state=42,
    ):
        self.keyword_column = keyword_column
        self.tagline_column = tagline_column
        self.overview_column = overview_column
        self.keyword_top_k = keyword_top_k
        self.tfidf_max_features = tfidf_max_features
        self.n_components = n_components
        self.random_state = random_state

    def _get_frame(self, X):
        columns = [self.keyword_column, self.tagline_column, self.overview_column]
        if isinstance(X, pd.DataFrame):
            return X.loc[:, columns].copy()
        return pd.DataFrame(X, columns=columns)

    def _prepare_keywords(self, frame):
        keywords = frame[self.keyword_column].fillna("").astype(str)
        keywords = keywords.where(keywords.ne("nan"), "")
        return keywords.str.replace("-", " ", regex=False)

    def _prepare_text_columns(self, frame):
        tagline = frame[self.tagline_column].fillna("").astype(str)
        overview = frame[self.overview_column].fillna("").astype(str)
        return tagline, overview

    def fit(self, X, y=None):
        frame = self._get_frame(X)
        keywords_clean = self._prepare_keywords(frame)
        tagline, overview = self._prepare_text_columns(frame)
        combined_text = tagline + " " + overview

        self.keyword_vectorizer_ = CountVectorizer(
            max_features=self.keyword_top_k,
            stop_words="english",
        )
        self.keyword_vectorizer_.fit(keywords_clean)
        self.keyword_feature_names_ = [
            f"kw_{feature_name}"
            for feature_name in self.keyword_vectorizer_.get_feature_names_out()
        ]

        self.tfidf_vectorizer_ = TfidfVectorizer(
            stop_words="english",
            max_features=self.tfidf_max_features,
        )
        text_tfidf_matrix = self.tfidf_vectorizer_.fit_transform(combined_text)

        self.svd_ = TruncatedSVD(
            n_components=self.n_components,
            random_state=self.random_state,
        )
        self.svd_.fit(text_tfidf_matrix)
        self.theme_feature_names_ = [f"text_theme_{i + 1}" for i in range(self.n_components)]

        self.feature_names_ = [
            "num_keywords",
            "overview_word_count",
            "tagline_word_count",
            *self.keyword_feature_names_,
            *self.theme_feature_names_,
        ]
        return self

    def transform(self, X):
        frame = self._get_frame(X)
        keywords_clean = self._prepare_keywords(frame)
        tagline, overview = self._prepare_text_columns(frame)
        combined_text = tagline + " " + overview

        meta_features = pd.DataFrame(
            {
                "num_keywords": keywords_clean.apply(lambda value: len(value.split()) if value else 0),
                "overview_word_count": overview.apply(lambda value: len(value.split())),
                "tagline_word_count": tagline.apply(lambda value: len(value.split())),
            },
            index=frame.index,
        )

        keyword_features = pd.DataFrame(
            self.keyword_vectorizer_.transform(keywords_clean).toarray(),
            columns=self.keyword_feature_names_,
            index=frame.index,
        )

        theme_features = pd.DataFrame(
            self.svd_.transform(self.tfidf_vectorizer_.transform(combined_text)),
            columns=self.theme_feature_names_,
            index=frame.index,
        )

        return pd.concat([meta_features, keyword_features, theme_features], axis=1)

    def get_feature_names_out(self, input_features=None):
        return np.array(self.feature_names_, dtype=object)


def build_multilabel_pipeline(config):
    return Pipeline(steps=[("encoder", MultiLabelColumnEncoder(**config))])


def build_onehot_pipeline(config):
    return Pipeline(steps=[("encoder", OneHotColumnEncoder(**config))])


def build_text_pipeline():
    return Pipeline(steps=[("encoder", MovieTextEncoder())])


def build_preprocessor():
    transformers = [("base_features", "passthrough", passthrough_features)]

    for column in multiLabelCategorical_features:
        transformers.append(
            (column, build_multilabel_pipeline(MULTILABEL_FEATURE_CONFIGS[column]), [column])
        )

    for column in OneHotEncodingCategorical_features:
        transformers.append(
            (column, build_onehot_pipeline(ONEHOT_FEATURE_CONFIGS[column]), [column])
        )

    transformers.append(("text_features", build_text_pipeline(), TextBased_features))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_feature_summary(preprocessor):
    summary_rows = []
    selected_features = {}

    for name, transformer, _ in preprocessor.transformers_:
        if name == "base_features" or not hasattr(transformer, "named_steps"):
            continue

        feature_names = transformer.named_steps["encoder"].get_feature_names_out().tolist()
        selected_features[name] = feature_names

        if name in multiLabelCategorical_features:
            encoding_name = "MultiLabelBinarizer"
        elif name in OneHotEncodingCategorical_features:
            encoding_name = "OneHotEncoder"
        else:
            encoding_name = "CountVectorizer + TfidfVectorizer + TruncatedSVD"

        summary_rows.append(
            {
                "feature": name,
                "encoding": encoding_name,
                "selected_columns": len(feature_names),
                "sample_columns": ", ".join(feature_names[:5]),
            }
        )

    return selected_features, pd.DataFrame(summary_rows)


In [2]:
df = pd.read_csv(DATA_PATH)

feature_pipeline = Pipeline(steps=[("preprocessor", build_preprocessor())])
feature_pipeline.set_output(transform="pandas")

df_encoded = feature_pipeline.fit_transform(df)
selected_features, encoding_summary_df = build_feature_summary(
    feature_pipeline.named_steps["preprocessor"]
)


In [3]:
df_encoded.to_csv(OUTPUT_PATH, index=False)
print(f"Saved encoded dataset to: {OUTPUT_PATH}")


Saved encoded dataset to: Dataset\comparison_outputs\Movies-Dataset-no-missing-2000-2026-ML-Ready-encoded.csv
